In [ ]:
# 1. 确保有 curl
!type -p curl >/dev/null || (apt update && apt install curl -y)

# 2. 下载 GitHub CLI 的密钥
!curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg

!chmod go+r /usr/share/keyrings/githubcli-archive-keyring.gpg

# 3. 添加软件源
!echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" | tee /etc/apt/sources.list.d/github-cli.list > /dev/null

# 4. 安装 gh
!apt update && apt install gh -y

8+1 records in
8+1 records out
4528 bytes (4.5 kB, 4.4 KiB) copied, 0.0968134 s, 46.8 kB/s
Get:1 https://cli.github.com/packages stable InRelease [3917 B]
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
0% [Connected to ppa.launchpadcontent.net (185.125.189.186)]               

In [ ]:
#GitHub
import os

REPO = "/workspace/stable-query-latent"
URL  = "https://github.com/Nice9Tian/stable-query-latent.git"

# 只在不存在时 clone；用绝对路径，不受当前 cwd 影响
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

# 永远 cd 到绝对路径，重跑多少次都不会嵌套
%cd {REPO}
!git remote set-url origin {URL}
!git remote -v
!git pull origin main

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('cxx11abi:', torch._C._GLIBCXX_USE_CXX11_ABI)
import sys; print(sys.version)

In [ ]:
!pip install flash-attn==2.7.4.post1 --no-build-isolation
import flash_attn
print("当前版本",flash_attn.__version__)        # 应输出 2.7.4.post1
from flash_attn import flash_attn_func   # 不报错 = 成功

In [ ]:
# 开启 GPU + CPU + RAM + Disk I/O 性能监视
import subprocess, threading, time, psutil
from pathlib import Path

stop = False

def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == "max":
            return None
        return int(text)
    except Exception:
        return None

def get_memory_status():
    # 优先读 Linux cgroup/container 限制
    limit = _read_int("/sys/fs/cgroup/memory.max")
    used = _read_int("/sys/fs/cgroup/memory.current")

    if limit is None or used is None:
        limit = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")

    if limit and used and limit < 10**18:
        used_gib = used / 1024**3
        total_gib = limit / 1024**3
        pct = used / limit * 100
        return pct, used_gib, total_gib, "cgroup"

    # fallback: psutil / host memory
    vm = psutil.virtual_memory()
    used_gib = vm.used / 1024**3
    total_gib = vm.total / 1024**3
    return vm.percent, used_gib, total_gib, "host"

def get_gpu_status():
    try:
        out = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()

        parts = [p.strip() for p in out.splitlines()[0].split(",")]
        util = float(parts[0])
        used_mib = float(parts[1])
        total_mib = float(parts[2])
        return f"{util:.0f}%, {used_mib/1024:.1f}/{total_mib/1024:.1f} GiB"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)

    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)

        ram_pct, ram_used, ram_total, ram_source = get_memory_status()

        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)

        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t

        print(
            f"[gpu] {gpu} | "
            f"[cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] {ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )

        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()

In [ ]:
#构建
!python -u game_review_data/build.py \
  --data-dir game_review_data/build_new_gamedata \
  --only metadata split text-h5 \
  --split-device cuda \
  --logout-address /workspace/stable_query_latent_logs/pipeline.log

In [ ]:
#清空显存
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#嵌入
!python -u game_review_data/embedding_incloud.py \
  --input-h5 game_review_data/build_new_gamedata/text_h5.h5 \
  --output-h5 game_review_data/embedding_h5.h5 \
  --device cuda \
  --text-load-chunk-size 500000 \
  --tok-workers 4 \
  --logout-address /workspace/stable_query_latent_logs/pipeline.log

In [ ]:
#清空显存
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#正式训练
!python -u VICReg_review/sweep_cloud.py \
  --h5 game_review_data/embedding_h5.h5 \
  --out-dir VICReg_review/heads/cloud_full_sweep_a100 \
  --train-game-counts 50 100 200 400 600 800 1000 1200 1400 1600 1800 2000\
  --sample-fractions 0.8 0.6 0.4 0.2 \
  --output-dims 18 36 64 72 \
  --epochs 30 \
  --steps-per-epoch 4 \
  --batch-size 128 \
  --backward-mode split_recompute \
  --cache-mode full \
  --prefetch-batches 2 \
  --cache-dtype float16 \
  --pin-cache \
  --train-probe-every 0 \
  --train-game-anchor-appids "1091500,1385380" \
  --logout-address /workspace/stable_query_latent_logs/pipeline.log

In [ ]:
#收集结果
stop = True

from pathlib import Path
from datetime import datetime
import json
import shutil

REPO = Path("/workspace/stable-query-latent")
if not REPO.exists():
    REPO = Path.cwd()

OUT = Path("/workspace") / "stable_query_latent_artifacts" / datetime.now().strftime("%Y%m%d_%H%M%S")
OUT.mkdir(parents=True, exist_ok=True)

def copy_item(rel: str):
    src = REPO / rel
    dst = OUT / rel
    if not src.exists():
        print(f"skip missing: {src}")
        return None

    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        kind = "dir"
    else:
        shutil.copy2(src, dst)
        kind = "file"

    print(f"copied: {rel}")
    return {"item": rel, "kind": kind}

items = [
    "VICReg_review/heads",
    "game_review_data/embedding_h5.h5",
    "game_review_data/embedding_h5.h5.incloud_manifest.json",
    "game_review_data/build_new_gamedata/text_h5.h5",
    "game_review_data/build_new_gamedata/text_h5.h5.manifest.json",
]

manifest = [x for x in (copy_item(rel) for rel in items) if x is not None]
(OUT / "collection_manifest.json").write_text(
    json.dumps({"repo": str(REPO), "archive": str(OUT), "items": manifest}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"archive ready: {OUT}")

In [ ]:
#清理
# from pathlib import Path
# import shutil

# root = Path("/workspace")
# for p in root.iterdir():
#     if p.name == "start.ipynb":
#         continue
#     if p.is_dir():
#         shutil.rmtree(p)
#     else:
#         p.unlink()